In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("messy_customer_dataset.csv")

# Display first 5 rows
print("First 5 Rows:")
display(df.head())

# Dataset shape
print("\nDataset Shape:", df.shape)

# Column names
print("\nColumn Names:")
print(df.columns.tolist())

# Data types
print("\nData Types:")
print(df.dtypes)

# Missing values
print("\nMissing Values per Column:")
print(df.isnull().sum())

# Duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())

# Basic statistics
print("\nNumerical Summary:")
display(df.describe(include="all"))

First 5 Rows:


,Customer_ID,Name,Gender,Age,City,Join_Date,Annual_Income,Purchase_Amount
0,C001,Aarav Shah,Male,25.0,Pune,2024-01-15,450000,1200.5
1,C002,Meera Patil,female,31.0,Mumbai,15/02/2024,620000,2500.0
2,C003,Rohan Deshmukh,M,29.0,PUNE,2024/03/10,510000,1800.0
3,C004,Sara Khan,F,NaN,Nashik,2024-04-02,580000,2100.0
4,C005,Kabir Joshi,Male,42.0,Aurangabad,2024-05-21,NaN,3200.0



Dataset Shape: (30, 8)

Column Names:
['Customer_ID', 'Name', 'Gender', 'Age', 'City', 'Join_Date', 'Annual_Income', 'Purchase_Amount']

Data Types:
Customer_ID            str
Name                   str
Gender                 str
Age                float64
City                   str
Join_Date              str
Annual_Income          str
Purchase_Amount    float64
dtype: object

Missing Values per Column:
Customer_ID        0
Name               0
Gender             0
Age                1
City               0
Join_Date          0
Annual_Income      1
Purchase_Amount    1
dtype: int64

Duplicate Rows: 0

Numerical Summary:


,Customer_ID,Name,Gender,Age,City,Join_Date,Annual_Income,Purchase_Amount
count,30,30,30,29.000000,30,30,29,29.000000
unique,30,29,6,NaN,8,29,27,NaN
top,C001,Aarav Shah,Male,NaN,Mumbai,2024-01-15,450000,NaN
freq,1,2,10,NaN,9,2,2,NaN
mean,NaN,NaN,NaN,34.724138,NaN,NaN,NaN,36567.241379
std,NaN,NaN,NaN,23.837091,NaN,NaN,NaN,185296.171110
min,NaN,NaN,NaN,-5.000000,NaN,NaN,NaN,-300.000000
25%,NaN,NaN,NaN,27.000000,NaN,NaN,NaN,1750.000000
50%,NaN,NaN,NaN,31.000000,NaN,NaN,NaN,2200.000000
75%,NaN,NaN,NaN,36.000000,NaN,NaN,NaN,2750.000000


In [2]:
# DATA QUALITY REPORT

print("========== DATA QUALITY REPORT ==========")

# 1. Missing values
print("\n1. Missing Values:")
print(df.isnull().sum())

# 2. Duplicate rows
print("\n2. Exact Duplicate Rows:")
print(df.duplicated().sum())

# 3. Duplicate customer IDs
print("\n3. Duplicate Customer IDs:")
print(df["Customer_ID"].duplicated().sum())

# 4. Invalid ages
age_numeric = pd.to_numeric(df["Age"], errors="coerce")
invalid_age = (age_numeric < 0) | (age_numeric > 100)

print("\n4. Invalid Age Records:")
display(df.loc[invalid_age, ["Customer_ID", "Name", "Age"]])

# 5. Invalid purchase amounts
purchase_numeric = pd.to_numeric(
    df["Purchase_Amount"], errors="coerce"
)

invalid_purchase = purchase_numeric < 0

print("\n5. Negative Purchase Amounts:")
display(df.loc[invalid_purchase, ["Customer_ID", "Name", "Purchase_Amount"]])

# 6. Invalid annual income values
income_numeric = pd.to_numeric(df["Annual_Income"], errors="coerce")

print("\n6. Non-numeric or Missing Annual Income:")
display(
    df.loc[income_numeric.isna(), ["Customer_ID", "Name", "Annual_Income"]]
)

# 7. Data types
print("\n7. Current Data Types:")
print(df.dtypes)

# 8. Suspiciously high purchase amounts
print("\n8. Purchase Amounts Above 10000:")
display(
    df.loc[purchase_numeric > 10000,
           ["Customer_ID", "Name", "Purchase_Amount"]]
)

print("\n========== REPORT COMPLETE ==========")

========== DATA QUALITY REPORT ==========

1. Missing Values:
Customer_ID        0
Name               0
Gender             0
Age                1
City               0
Join_Date          0
Annual_Income      1
Purchase_Amount    1
dtype: int64

2. Exact Duplicate Rows:
0

3. Duplicate Customer IDs:
0

4. Invalid Age Records:


,Customer_ID,Name,Age
14,C015,Atharv Shinde,150.0
25,C026,Riya Nair,-5.0



5. Negative Purchase Amounts:


,Customer_ID,Name,Purchase_Amount
25,C026,Riya Nair,-300.0



6. Non-numeric or Missing Annual Income:


,Customer_ID,Name,Annual_Income
4,C005,Kabir Joshi,NaN
25,C026,Riya Nair,unknown



7. Current Data Types:
Customer_ID            str
Name                   str
Gender                 str
Age                float64
City                   str
Join_Date              str
Annual_Income          str
Purchase_Amount    float64
dtype: object

8. Purchase Amounts Above 10000:


,Customer_ID,Name,Purchase_Amount
16,C017,Omkar Patil,999999.0



========== REPORT COMPLETE ==========


In [4]:
# Create a copy of the original dataset
df_clean = df.copy()

# 1. Convert columns to numeric
df_clean["Age"] = pd.to_numeric(df_clean["Age"], errors="coerce")
df_clean["Annual_Income"] = pd.to_numeric(
    df_clean["Annual_Income"], errors="coerce"
)
df_clean["Purchase_Amount"] = pd.to_numeric(
    df_clean["Purchase_Amount"], errors="coerce"
)

# 2. Handle invalid ages
df_clean.loc[
    (df_clean["Age"] < 0) | (df_clean["Age"] > 100), "Age"
] = np.nan

# 3. Handle negative purchase amounts
df_clean.loc[df_clean["Purchase_Amount"] < 0, "Purchase_Amount"] = np.nan

# 4. Convert dates
df_clean["Join_Date"] = pd.to_datetime(
    df_clean["Join_Date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)

# 5. Standardise Gender
df_clean["Gender"] = (
    df_clean["Gender"].astype("string").str.strip().str.lower()
)

gender_mapping = {
    "m": "Male",
    "male": "Male",
    "f": "Female",
    "female": "Female"
}

df_clean["Gender"] = df_clean["Gender"].map(gender_mapping)

# 6. Standardise City names
df_clean["City"] = (
    df_clean["City"].astype("string").str.strip().str.title()
)

# 7. Fill missing values
df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())

df_clean["Annual_Income"] = df_clean["Annual_Income"].fillna(
    df_clean["Annual_Income"].median()
)

df_clean["Purchase_Amount"] = df_clean["Purchase_Amount"].fillna(
    df_clean["Purchase_Amount"].median()
)

df_clean["Gender"] = df_clean["Gender"].fillna(
    df_clean["Gender"].mode()[0]
)

# 8. Display results
print("Cleaned Dataset Preview:")
display(df_clean.head(10))

print("\nMissing Values After Cleaning:")
print(df_clean.isnull().sum())

print("\nUpdated Data Types:")
print(df_clean.dtypes)

Cleaned Dataset Preview:


,Customer_ID,Name,Gender,Age,City,Join_Date,Annual_Income,Purchase_Amount
0,C001,Aarav Shah,Male,25.0,Pune,2024-01-15,450000.0,1200.5
1,C002,Meera Patil,Female,31.0,Mumbai,2024-02-15,620000.0,2500.0
2,C003,Rohan Deshmukh,Male,29.0,Pune,2024-10-03,510000.0,1800.0
3,C004,Sara Khan,Female,31.0,Nashik,2024-02-04,580000.0,2100.0
4,C005,Kabir Joshi,Male,42.0,Aurangabad,2024-05-21,585000.0,3200.0
5,C006,Anaya Kulkarni,Female,35.0,Pune,2024-06-21,720000.0,2750.0
6,C007,Ishaan More,Male,27.0,Mumbai,2024-11-07,490000.0,1500.0
7,C008,Diya Pawar,Female,30.0,Nagpur,2024-09-08,550000.0,1900.0
8,C009,Vivaan Jadhav,Male,33.0,Nashik,2024-09-18,610000.0,2300.0
9,C010,Ira Sane,Female,28.0,Pune,2024-05-10,530000.0,1750.0



Missing Values After Cleaning:
Customer_ID        0
Name               0
Gender             0
Age                0
City               0
Join_Date          2
Annual_Income      0
Purchase_Amount    0
dtype: int64

Updated Data Types:
Customer_ID                   str
Name                          str
Gender                        str
Age                       float64
City                       string
Join_Date          datetime64[us]
Annual_Income             float64
Purchase_Amount           float64
dtype: object


In [6]:
# 1. Correct date formats
date_text = df["Join_Date"].astype("string").str.strip()

parsed_dates = pd.to_datetime(
    date_text, format="%Y-%m-%d", errors="coerce"
)

mask = parsed_dates.isna()
parsed_dates.loc[mask] = pd.to_datetime(
    date_text.loc[mask], format="%d/%m/%Y", errors="coerce"
)

mask = parsed_dates.isna()
parsed_dates.loc[mask] = pd.to_datetime(
    date_text.loc[mask], format="%Y/%m/%d", errors="coerce"
)

mask = parsed_dates.isna()
parsed_dates.loc[mask] = pd.to_datetime(
    date_text.loc[mask], format="%d-%m-%Y", errors="coerce"
)

df_clean["Join_Date"] = parsed_dates

# 2. Count and remove exact duplicate rows
duplicates_before = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().copy()

print("Exact duplicate rows removed:", duplicates_before)
print("Rows after duplicate removal:", len(df_clean))

# 3. Display records with invalid dates
print("\nRecords with missing or invalid dates:")
display(
    df_clean.loc[
        df_clean["Join_Date"].isna(),
        ["Customer_ID", "Name", "Join_Date"]
    ]
)

# 4. Display date range
print("\nEarliest date:", df_clean["Join_Date"].min())
print("Latest date:", df_clean["Join_Date"].max())

Exact duplicate rows removed: 0
Rows after duplicate removal: 30

Records with missing or invalid dates:


,Customer_ID,Name,Join_Date
25,C026,Riya Nair,NaT
26,C027,Vedant Kale,NaT



Earliest date: 2024-01-15 00:00:00
Latest date: 2025-12-20 00:00:00


In [7]:
# Detect outliers using the IQR method

numeric_columns = ["Age", "Annual_Income", "Purchase_Amount"]

outlier_summary = []

for column in numeric_columns:
    Q1 = df_clean[column].quantile(0.25)
    Q3 = df_clean[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = df_clean[
        (df_clean[column] < lower_limit) |
        (df_clean[column] > upper_limit)
    ]

    outlier_summary.append({
        "Column": column,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower Limit": lower_limit,
        "Upper Limit": upper_limit,
        "Outlier Count": len(outliers)
    })

    print("\nColumn:", column)
    print("Lower Limit:", round(lower_limit, 2))
    print("Upper Limit:", round(upper_limit, 2))
    print("Outlier Count:", len(outliers))

    display(outliers[["Customer_ID", "Name", column]])

# Display summary table
outlier_report = pd.DataFrame(outlier_summary)

print("\nOUTLIER SUMMARY")
display(outlier_report)


Column: Age
Lower Limit: 17.88
Upper Limit: 44.88
Outlier Count: 1


,Customer_ID,Name,Age
12,C013,Reyansh Kale,45.0



Column: Annual_Income
Lower Limit: 350000.0
Upper Limit: 810000.0
Outlier Count: 1


,Customer_ID,Name,Annual_Income
12,C013,Reyansh Kale,850000.0



Column: Purchase_Amount
Lower Limit: 462.5
Upper Limit: 4062.5
Outlier Count: 2


,Customer_ID,Name,Purchase_Amount
12,C013,Reyansh Kale,4100.0
16,C017,Omkar Patil,999999.0



OUTLIER SUMMARY


,Column,Q1,Q3,IQR,Lower Limit,Upper Limit,Outlier Count
0,Age,28.0,34.75,6.75,17.875,44.875,1
1,Annual_Income,522500.0,637500.00,115000.00,350000.000,810000.000,1
2,Purchase_Amount,1812.5,2712.50,900.00,462.500,4062.500,2


In [8]:
# Handle outliers using the IQR method

Q1 = df_clean["Purchase_Amount"].quantile(0.25)
Q3 = df_clean["Purchase_Amount"].quantile(0.75)
IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Purchase Amount Lower Limit:", lower_limit)
print("Purchase Amount Upper Limit:", upper_limit)

# Cap extreme purchase amounts at the IQR limits
df_clean["Purchase_Amount"] = df_clean["Purchase_Amount"].clip(
    lower=lower_limit,
    upper=upper_limit
)

print("\nPurchase Amount after outlier handling:")
display(
    df_clean.loc[
        df_clean["Customer_ID"] == "C017",
        ["Customer_ID", "Name", "Purchase_Amount"]
    ]
)

Purchase Amount Lower Limit: 462.5
Purchase Amount Upper Limit: 4062.5

Purchase Amount after outlier handling:


,Customer_ID,Name,Purchase_Amount
16,C017,Omkar Patil,4062.5


In [9]:
# Step 10: Final data validation

# Convert columns to correct data types
df_clean["Customer_ID"] = df_clean["Customer_ID"].astype("string")
df_clean["Name"] = df_clean["Name"].astype("string")
df_clean["Gender"] = df_clean["Gender"].astype("string")
df_clean["City"] = df_clean["City"].astype("string")

df_clean["Age"] = df_clean["Age"].round().astype("Int64")
df_clean["Annual_Income"] = df_clean["Annual_Income"].round().astype("Int64")
df_clean["Purchase_Amount"] = df_clean["Purchase_Amount"].round(2)
df_clean["Join_Date"] = pd.to_datetime(df_clean["Join_Date"], errors="coerce")

print("Dataset Shape:", df_clean.shape)

print("\nMissing Values:")
print(df_clean.isnull().sum())

print("\nData Types:")
print(df_clean.dtypes)

print("\nDuplicate Rows:", df_clean.duplicated().sum())

display(df_clean.head(10))

Dataset Shape: (30, 8)

Missing Values:
Customer_ID        0
Name               0
Gender             0
Age                0
City               0
Join_Date          2
Annual_Income      0
Purchase_Amount    0
dtype: int64

Data Types:
Customer_ID                string
Name                       string
Gender                     string
Age                         Int64
City                       string
Join_Date          datetime64[us]
Annual_Income               Int64
Purchase_Amount           float64
dtype: object

Duplicate Rows: 0


,Customer_ID,Name,Gender,Age,City,Join_Date,Annual_Income,Purchase_Amount
0,C001,Aarav Shah,Male,25,Pune,2024-01-15,450000,1200.5
1,C002,Meera Patil,Female,31,Mumbai,2024-02-15,620000,2500.0
2,C003,Rohan Deshmukh,Male,29,Pune,2024-03-10,510000,1800.0
3,C004,Sara Khan,Female,31,Nashik,2024-04-02,580000,2100.0
4,C005,Kabir Joshi,Male,42,Aurangabad,2024-05-21,585000,3200.0
5,C006,Anaya Kulkarni,Female,35,Pune,2024-06-21,720000,2750.0
6,C007,Ishaan More,Male,27,Mumbai,2024-07-11,490000,1500.0
7,C008,Diya Pawar,Female,30,Nagpur,2024-08-09,550000,1900.0
8,C009,Vivaan Jadhav,Male,33,Nashik,2024-09-18,610000,2300.0
9,C010,Ira Sane,Female,28,Pune,2024-10-05,530000,1750.0


In [10]:
# Fill missing dates using the median date
valid_dates = df_clean["Join_Date"].dropna().sort_values()

median_date = valid_dates.iloc[len(valid_dates) // 2]

df_clean["Join_Date"] = df_clean["Join_Date"].fillna(median_date)

print("Missing values after date cleaning:")
print(df_clean.isnull().sum())

print("\nDates filled using:", median_date)

Missing values after date cleaning:
Customer_ID        0
Name               0
Gender             0
Age                0
City               0
Join_Date          0
Annual_Income      0
Purchase_Amount    0
dtype: int64

Dates filled using: 2025-02-17 00:00:00


In [11]:
# Before and after cleaning comparison

comparison = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Missing Values",
        "Duplicate Rows",
        "Invalid Age Values",
        "Negative Purchase Amounts"
    ],
    "Before Cleaning": [
        len(df),
        len(df.columns),
        int(df.isnull().sum().sum()),
        int(df.duplicated().sum()),
        int(((pd.to_numeric(df["Age"], errors="coerce") < 0) |
             (pd.to_numeric(df["Age"], errors="coerce") > 100)).sum()),
        int((pd.to_numeric(df["Purchase_Amount"], errors="coerce") < 0).sum())
    ],
    "After Cleaning": [
        len(df_clean),
        len(df_clean.columns),
        int(df_clean.isnull().sum().sum()),
        int(df_clean.duplicated().sum()),
        int(((df_clean["Age"] < 0) | (df_clean["Age"] > 100)).sum()),
        int((df_clean["Purchase_Amount"] < 0).sum())
    ]
})

display(comparison)

,Metric,Before Cleaning,After Cleaning
0,Total Rows,30,30
1,Total Columns,8,8
2,Missing Values,3,0
3,Duplicate Rows,0,0
4,Invalid Age Values,2,0
5,Negative Purchase Amounts,1,0


In [12]:
# Step 13: Save the cleaned dataset

output_file = "cleaned_customer_dataset.csv"

df_clean.to_csv(output_file, index=False)

print("Cleaned dataset saved successfully!")
print("File name:", output_file)
print("Final dataset shape:", df_clean.shape)
print("Total missing values:", df_clean.isnull().sum().sum())
print("Total duplicate rows:", df_clean.duplicated().sum())

Cleaned dataset saved successfully!
File name: cleaned_customer_dataset.csv
Final dataset shape: (30, 8)
Total missing values: 0
Total duplicate rows: 0


In [13]:
# Save the notebook's cleaned dataset as a CSV file
df_clean.to_csv("cleaned_customer_dataset.csv", index=False)

print("Cleaned CSV saved successfully!")
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])

Cleaned CSV saved successfully!
Rows: 30
Columns: 8
